# Senza guardare i polmoni

**CAF GIAS, Universita degli Studi Link, modulo 2.4** | Michele Ferramola

Dimostrazione da 5 minuti per il caso 2, *Le condizioni di acquisizione*.

---

### La domanda

Un archivio di radiografie del torace. Se **non guardo le immagini** e tengo solo
*come* e stata fatta la lastra, quanto riesco a sapere del reperto?

### Il dato

NIH ChestX-ray14 (NIH Clinical Center): 112.120 radiografie del torace di 30.805 pazienti.
Di ogni radiografia usiamo solo i metadati: eta, sesso, proiezione (PA in piedi, AP a letto),
dimensioni e risoluzione dell'immagine, che identificano l'apparecchio.
I reperti sono estratti automaticamente dai referti, quindi contengono errori.

Uso libero citando Wang X. e colleghi, *ChestX-ray8*, IEEE CVPR 2017.
Dati originali: https://nihcc.app.box.com/v/ChestXray-NIHCC

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

URL = ("https://huggingface.co/datasets/alkzar90/NIH-Chest-X-ray-dataset/resolve/main/"
       "data/Data_Entry_2017_v2020.csv")
d = pd.read_csv(URL)
d.columns = ["immagine", "reperti", "controllo_n", "paziente", "eta", "sesso", "proiezione",
             "larghezza", "altezza", "pixel_x", "pixel_y"]
d["AP"] = (d.proiezione == "AP").astype(int)          # 1 = lastra fatta a letto
d["donna"] = (d.sesso == "F").astype(int)
d["edema"] = d.reperti.str.contains("Edema").astype(int)

migliaia = lambda n: f"{n:,}".replace(",", ".")
print(f"radiografie: {migliaia(len(d))}   pazienti: {migliaia(d.paziente.nunique())}   "
      f"fatte a letto: {d.AP.mean():.0%}")
d[["eta", "sesso", "proiezione", "larghezza", "pixel_x", "reperti"]].head()

Nessuna immagine: solo come e stata fatta la lastra, e a chi.

### Come e stata fatta la lastra

La proiezione AP si fa a letto, a chi non puo alzarsi: i pazienti piu gravi.

In [ ]:
voci = {"No Finding": "nessun reperto", "Effusion": "versamento pleurico", "Pneumonia": "polmonite",
        "Consolidation": "consolidamento", "Edema": "edema"}
quota = pd.Series({it: d.loc[(d.reperti == en) if en == "No Finding" else d.reperti.str.contains(en), "AP"].mean()
                   for en, it in voci.items()})
ax = quota.plot.barh(figsize=(8, 3.6), color=["#9a9892", "#2a78d6", "#2a78d6", "#2a78d6", "#b07a0a"])
for i, v in enumerate(quota):
    ax.text(v + 0.01, i, f"{v:.0%}", va="center")
ax.set_xlim(0, 1.05); ax.set_xlabel("quota di radiografie fatte a letto (AP)")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("La proiezione tradisce il reperto"); plt.tight_layout(); plt.show()

L'88% delle radiografie con edema e fatto a letto, contro il 35% di quelle senza reperti.

### Adesso la prova

Un modello cerca l'edema **senza vedere l'immagine**.
Lo alleno su una parte dei pazienti e lo verifico su pazienti che non ha mai visto:
nessun paziente sta in entrambe le parti.

In [ ]:
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0).split(d, groups=d.paziente))
A, B = d.iloc[tr], d.iloc[te]
print(f"verifica: {migliaia(len(B))} radiografie di {migliaia(B.paziente.nunique())} pazienti, "
      f"{B.edema.sum()} con edema")
print()
print(f"solo la proiezione                     AUROC = {roc_auc_score(B.edema, B.AP):.3f}")

colonne = ["AP", "eta", "donna", "larghezza", "altezza", "pixel_x", "pixel_y"]
mod = HistGradientBoostingClassifier(max_iter=200, random_state=0).fit(A[colonne], A.edema)
print(f"proiezione, eta, sesso e apparecchio   AUROC = {roc_auc_score(B.edema, mod.predict_proba(B[colonne])[:, 1]):.3f}")

Senza un pixel dei polmoni, il modello riconosce l'edema molto meglio del caso:
ha imparato la logistica del reparto, non la malattia.

### Per gruppo, con quanti pazienti

Stessa verifica, separata per sesso e per eta. Accanto al risultato, quanti casi di edema lo sostengono.

In [ ]:
gruppi = {"donne": B.donna == 1, "uomini": B.donna == 0,
          "sotto i 30 anni": B.eta < 30, "30-49 anni": B.eta.between(30, 49),
          "50-69 anni": B.eta.between(50, 69), "70 anni e oltre": B.eta >= 70}
righe = [(nome, len(B[m]), B[m].edema.sum(), roc_auc_score(B[m].edema, B[m].AP)) for nome, m in gruppi.items()]
pd.DataFrame(righe, columns=["gruppo", "radiografie", "casi di edema", "AUROC della sola proiezione"]).round(2)

La stessa scorciatoia rende in modo diverso da un gruppo all'altro, perche le lastre a letto
non si distribuiscono allo stesso modo. E sopra i 70 anni il numero poggia su pochi casi:
senza sapere quanti pazienti ci sono in ogni gruppo, un risultato per gruppo non si puo giudicare.

### Quando dice edema, ha ragione?

In [ ]:
print(f"radiografie di verifica con edema:           {B.edema.mean():.1%}")
print(f"fra quelle fatte a letto, quelle con edema:  {B.loc[B.AP == 1, 'edema'].mean():.1%}")

---

## Cosa abbiamo visto

1. **L'indizio sta nel modo in cui e stata fatta la lastra.** La proiezione a letto porta con se
   la gravita del paziente, e un modello la usa senza guardare i polmoni.
2. **La scorciatoia non vale uguale per tutti.** Cambia fra sessi e fasce d'eta, e con pochi casi
   il risultato non si puo giudicare.
3. **Il valore predittivo resta basso**, perche l'edema e raro: e il tema del caso 3.

### La verifica da portare in gara

> **Da quali ospedali, con quali apparecchi e protocolli? Su quanti ospedali e stato verificato?**